En este notebook vienen alguns ejemplos de como obtener y convrtir datos en tensores usando pytorch

1. Datos en tablas (un ejemplo de características químicas de vinos)

In [9]:
import numpy as np
import csv          #Para datos en formato .csv
import torch

#Definimos una variable con la ruta del acrhivo en cadena de caracteres
wine_path = "winequality-white.csv"

#Definimos un arreglo de numpy leyendo solo los datos del archivo
wine_np = np.loadtxt(wine_path, dtype=np.float32, delimiter=";", skiprows=1)
#loadtxt es una función de numpy para leer archivos de texto, lee: la ruta, el tipo de dato, delimitador,
#y en este caso saltamos el encabezado

wine_np, wine_np.shape

(array([[ 7.  ,  0.27,  0.36, ...,  0.45,  8.8 ,  6.  ],
        [ 6.3 ,  0.3 ,  0.34, ...,  0.49,  9.5 ,  6.  ],
        [ 8.1 ,  0.28,  0.4 , ...,  0.44, 10.1 ,  6.  ],
        ...,
        [ 6.5 ,  0.24,  0.19, ...,  0.46,  9.4 ,  6.  ],
        [ 5.5 ,  0.29,  0.3 , ...,  0.38, 12.8 ,  7.  ],
        [ 6.  ,  0.21,  0.38, ...,  0.32, 11.8 ,  6.  ]], dtype=float32),
 (4898, 12))

In [7]:
#Enlistamos por separado las etiquetas de las columnas
col_list = next(csv.reader(open(wine_path), delimiter=';'))
#open es el comando usual para abrir un archivo, csv.reader lee el archivo tipo .csv con delimitador ;
#esto genera una lista de elementos correspondiente a las filas del archivo, por último next lee este item

col_list

['fixed acidity',
 'volatile acidity',
 'citric acid',
 'residual sugar',
 'chlorides',
 'free sulfur dioxide',
 'total sulfur dioxide',
 'density',
 'pH',
 'sulphates',
 'alcohol',
 'quality']

In [11]:
#Convertimos el arreglo numpy a tensor en pytorch
wine = torch.from_numpy(wine_np)

wine.shape, wine.dtype

(torch.Size([4898, 12]), torch.float32)

In [13]:
#Podemos separar las características del vino de las calificaciones
data =  wine[:, :-1]   #todos los renglones y todas las columnas menos la última
data, data.shape

(tensor([[ 7.0000,  0.2700,  0.3600,  ...,  3.0000,  0.4500,  8.8000],
         [ 6.3000,  0.3000,  0.3400,  ...,  3.3000,  0.4900,  9.5000],
         [ 8.1000,  0.2800,  0.4000,  ...,  3.2600,  0.4400, 10.1000],
         ...,
         [ 6.5000,  0.2400,  0.1900,  ...,  2.9900,  0.4600,  9.4000],
         [ 5.5000,  0.2900,  0.3000,  ...,  3.3400,  0.3800, 12.8000],
         [ 6.0000,  0.2100,  0.3800,  ...,  3.2600,  0.3200, 11.8000]]),
 torch.Size([4898, 11]))

In [16]:
target = wine[:, -1].long()  #el método .long() de torch cambia real a entero
target

tensor([6, 6, 6,  ..., 6, 7, 6])

2. Series de tiempo (un ejemplo de datos de clima y ocupación de bicicletas por hora)

In [21]:
#Escribimos la variable del archivo
bike_path = "hour.csv"

#Hacemos el arreglo numpy como antes
bike_np = np.loadtxt(bike_path, dtype=np.float32, delimiter=",", skiprows=1, converters={1: lambda x: float(x[8:10])})
#converters lo que hace es aplicar una regla de transformación específica para cierta columna
#la función lambda convierte a flotante el valor del día en la fecha con el formato yyyy-mm-dd

#Convertimos el arreglo a tensor
bikes = torch.from_numpy(bike_np)
bikes, bikes.shape

(tensor([[1.0000e+00, 1.0000e+00, 1.0000e+00,  ..., 3.0000e+00, 1.3000e+01,
          1.6000e+01],
         [2.0000e+00, 1.0000e+00, 1.0000e+00,  ..., 8.0000e+00, 3.2000e+01,
          4.0000e+01],
         [3.0000e+00, 1.0000e+00, 1.0000e+00,  ..., 5.0000e+00, 2.7000e+01,
          3.2000e+01],
         ...,
         [1.7377e+04, 3.1000e+01, 1.0000e+00,  ..., 7.0000e+00, 8.3000e+01,
          9.0000e+01],
         [1.7378e+04, 3.1000e+01, 1.0000e+00,  ..., 1.3000e+01, 4.8000e+01,
          6.1000e+01],
         [1.7379e+04, 3.1000e+01, 1.0000e+00,  ..., 1.2000e+01, 3.7000e+01,
          4.9000e+01]]),
 torch.Size([17379, 17]))

In [26]:
bikes.stride(), bikes.shape[1]

((17, 1), 17)

In [30]:
#Como el número de horas (filas) no coincide con una cantidad entera de días vamos a cortar la tabla
n_days = bikes.shape[0] // 24      #Esto nos da el número entero de días que tenemos
bikes_cut = bikes[:n_days * 24,:]  #Esto hace el corte de nuestra tabla original

#Si queremos tener una serie de tiempo por día, por ejemplo, modificamos el tensor así
daily_bikes = bikes_cut.view(-1, 24, bikes_cut.shape[1])
#Esto da un tensor donde agrupa el original en grupos de 24 horas con las columnas (-1 hace el ajuste automático)
daily_bikes.shape, daily_bikes.stride()

(torch.Size([724, 24, 17]), (408, 17, 1))

Hasta ahora hemos tratado con datos numéricos, algunos continuos y otros solamente ordinales. Sin embargo, también existen los categóricos y estos se suelen codificar de la manera:
![](Datos.png)

3. Imágenes (un ejemplo de una imagen y un conjunto de imágenes .png)

In [39]:
import imageio.v3 as iio     ##modulo para importar imagenes como arreglos (también se puede usar torch vision)

#Importamos la imagen com arreglo
img_arr = iio.imread("Datos.png") 
img_arr.shape

(517, 856, 3)

Esta imagne fue imporatada como H x W x C, H=height, W=width, C=chanel.
Sin embargo, Torch trabaja las imagenes en el formato C x H x W. Para esto hay dos formas de trabajar la imagen, modificando el arreglo o importando directamente con TorchVison que lo hace automáticamente (en tensorflow se acomoda de manera usual H x W x C).

In [36]:
#Cambiando el formato
img_pre = torch.from_numpy(img_arr)
img = img_pre.permute(2, 0, 1)
img.shape, img.dtype

(torch.Size([3, 517, 856]), torch.uint8)

In [37]:
#Importando directamente con torch
from torchvision import io       #Modulo torch

img_torch = io.read_image("Datos.png")  # Lee la imagen como tensor uint8 (C, H, W)
img_torch.shape, img.dtype

(torch.Size([3, 517, 856]), torch.uint8)

Cuando tenemos un lote de imágenes del mismo tamaño en lugar de cargarlas como tensores y después juntarlas en un nuevo tensor, podemos desde el principio crear el tensor y cargarlas en la forma N x C x H x W, N=no.imagen

In [38]:
batch_size = 3    #Número total de imágenes
batch = torch.zeros(batch_size, 3, 256, 256, dtype=torch.uint8)
#Es un tensor con imágenes a color de 256x256

In [ ]:
import os                         #modulo para acceder a directorios y el sistema en general

data_dir = '../data/images/'      #ruta de la carpeta de imágenes

filenames = [name for name in os.listdir(data_dir)
            if os.path.splitext(name)[-1] == '.png']
#Crea una lista con los nombres de las imágenes .png

#Ahora leer las imágenes una por una e irla guardando en batch
for i, filenames in enumerate(filenames):
    img_arr = iio.imread(os.path.join(data_dir, filename))
    img_t = torch.from_numpy(img_arr)
    img_t = img_t.permute(2, 0, 1)
    img_t = img_t[:3]
    batch[i] = img_t

4. Imágenes 3D (un ejemplo de imagenes 3D de tomografías computarizadas)

In [ ]:
import imageio

dir_path = "volumetric-dicom/2-LUNG 3.0 B70f-04083"
vol_arr = iio.volread(dir_path, 'DICOM')
vol_arr.shape

En este caso tenemos una imagen en la forma D x H x W, falta agregar la información de color

In [ ]:
vol = torch.from_numpy(vol_arr).float()
vol = torch.unsqueeze(vol, 0)

Ahora tenemos la imágen comleta C x D x H x W
Y si tenemos un conjunto de imágenes tendremos un tensor N x C x D x H x W

5. Texto (un ejemplo con el libro orgullo y prefuicio)

In [47]:
#Abrimos el archivo de manera usual
with open('1342-0.txt', encoding='utf8') as file:
    text = file.read()
    #sacamos el texto como cadena de caracteres

#Separamos el texto línea por línea
lines = text.split('\n')
line = lines[349]
line

'Collins, when she draws the moral of Lydia’s fall. I sometimes wish'

In [50]:
#Utilizamos el método de crear el tensor y después almacenar con el método one-hot
letter_t = torch.zeros(len(line), 128)    #Almacenará las letras por su posición y código ascii en vectores renglón unitarios
letter_t.shape, letter_t

(torch.Size([67, 128]),
 tensor([[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]))

In [52]:
#Guardamos el texto en el tensor
for i, letter in enumerate(line.lower().strip()):
    letter_index = ord(letter) if ord(letter) < 128 else 0   #Condición de existencia del código
    letter_t[i][letter_index] = 1

letter_t

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [56]:
def clean_words(input_str):
    punctuation = '.,;:"!?´¸¨`^’'
    word_list = input_str.lower().replace('\n', ' ').split()    #Quita saltos de línea y espacios de la cadena
    word_list = [word.strip(punctuation) for word in word_list]  #Quita los signos de puntuación
    return word_list

words_in_line = clean_words(line)       #limpia la línea que elegimos
line, words_in_line

('Collins, when she draws the moral of Lydia’s fall. I sometimes wish',
 ['collins',
  'when',
  'she',
  'draws',
  'the',
  'moral',
  'of',
  'lydia’s',
  'fall',
  'i',
  'sometimes',
  'wish'])

In [57]:
#Ahora limpiamos todo el texto
word_list = sorted(set(clean_words(text)))
#Creamos el diccionario de palabras
word2index_dict = {word: i for (i, word) in enumerate(word_list)}

len(word2index_dict), word2index_dict['impossible']

(8999, 4123)

In [58]:
#Convertimos a tensor una palabara en la línea que vimos antes
word_t = torch.zeros(len(words_in_line), len(word2index_dict))    #Creamos el tensor
for i, word in enumerate(words_in_line):                           #Lo reyenamos con el método one-hot
    word_index = word2index_dict[word]
    word_t[i][word_index] = 1
    print('{:2} {:4} {}'.format(i, word_index, word))

print(word_t.shape)

 0 1624 collins
 1 8440 when
 2 7042 she
 3 2604 draws
 4 7711 the
 5 5187 moral
 6 5481 of
 7 4880 lydia’s
 8 3102 fall
 9 4028 i
10 7247 sometimes
11 8511 wish
torch.Size([12, 8999])


Otra forma de codificar texto es usando encajes la cual depende de una matriz de encaje como se ve en enesta imagen:

![](Embedding.png)